# AIOps Score Silver

Purpose: score the current silver standardized header and line datasets with the published PyTorch autoencoders. 


In [0]:
%pip install torch --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json, os
from datetime import datetime, timezone
import numpy as np
import torch
from torch import nn
from pyspark.sql import functions as F, types as T

STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

dbutils.widgets.text('model_version', 'v1')
MODEL_VERSION = dbutils.widgets.get('model_version')
BASE = f'abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net'
RUN_ID = spark.sql("select date_format(current_timestamp(), 'yyyyMMddHHmmss') run_id").first()['run_id']
NOTEBOOK_STARTED_AT = datetime.now(timezone.utc)

MODEL_STORAGE_ROOT = f'{BASE}/monitoring/aiops/model_registry/{MODEL_VERSION}'
LOCAL_MODEL_ROOT = f'/dbfs/tmp/aiops_model_registry_scoring/{MODEL_VERSION}'
DBFS_MODEL_ROOT = f'dbfs:/tmp/aiops_model_registry_scoring/{MODEL_VERSION}'

SILVER_HEADER_PATH = f'{BASE}/silver/transform/standardized/header/'
SILVER_LINES_PATH = f'{BASE}/silver/transform/standardized/lines/'
HEADER_SCORE_PATH = f'{BASE}/monitoring/aiops/scores/silver_header/'
LINES_SCORE_PATH = f'{BASE}/monitoring/aiops/scores/silver_lines/'
SCORING_HEADER_FEATURE_PATH = f'{BASE}/monitoring/aiops/features/scoring/silver_header/'
SCORING_LINES_FEATURE_PATH = f'{BASE}/monitoring/aiops/features/scoring/silver_lines/'
AIOPS_FEATURE_CONTRIBUTION_PATH = f'{BASE}/monitoring/aiops/feature_contributions/silver/'
AIOPS_ISSUE_LOG_PATH = f'{BASE}/monitoring/aiops/issue_log/'
AIOPS_WARNING_HEADER_PATH = f'{BASE}/silver/aiops/warning/header/'
AIOPS_WARNING_LINES_PATH = f'{BASE}/silver/aiops/warning/lines/'
AIOPS_QUARANTINE_HEADER_PATH = f'{BASE}/silver/aiops/quarantine/header/'
AIOPS_QUARANTINE_LINES_PATH = f'{BASE}/silver/aiops/quarantine/lines/'

AIOPS_RUNTIME_METRICS_PATH = f'{BASE}/monitoring/aiops_runtime_metrics/'
AIOPS_RUNTIME_METRICS_TABLE = '`hant-catalog`.invoice.batch_aiops_runtime_metrics'
AIOPS_RUNTIME_SCHEMA = T.StructType([
    T.StructField('layer', T.StringType(), False),
    T.StructField('operation_name', T.StringType(), False),
    T.StructField('run_id', T.StringType(), True),
    T.StructField('model_version', T.StringType(), True),
    T.StructField('started_at', T.TimestampType(), False),
    T.StructField('ended_at', T.TimestampType(), False),
    T.StructField('runtime_seconds', T.DoubleType(), False),
    T.StructField('rows_evaluated', T.LongType(), True),
    T.StructField('anomalies_detected', T.LongType(), True),
    T.StructField('records_per_second', T.DoubleType(), True),
    T.StructField('recorded_at', T.TimestampType(), False),
])

dbutils.fs.rm(DBFS_MODEL_ROOT, True) # Clean up any existing model in DBFS to ensure we have the latest version for scoring
dbutils.fs.cp(MODEL_STORAGE_ROOT, DBFS_MODEL_ROOT, True) # Copy the model from the storage to DBFS for loading with PyTorch

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8d250a9f-9c27-4071-8ebe-8098694f565a/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


True

In [0]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        latent_dim = max(2, min(8, input_dim // 2))
        hidden_dim = max(8, input_dim * 2)
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, latent_dim), nn.ReLU(), nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim))
    def forward(self, x):
        return self.net(x)

def load_model(model_name):
    model_dir = os.path.join(LOCAL_MODEL_ROOT, model_name)
    with open(os.path.join(model_dir, 'metadata.json'), 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    model = Autoencoder(metadata['input_dim'])
    model.load_state_dict(torch.load(os.path.join(model_dir, 'model.pt'), map_location='cpu'))
    model.eval()
    return model, metadata

def read_delta(path, label):
    df = spark.read.format('delta').load(path)
    if df.rdd.isEmpty():
        raise RuntimeError(f'{label} is empty: {path}')
    return df

def num(df, name):
    return F.col(name).cast('double') if name in df.columns else F.lit(None).cast('double')

def present(df, name):
    return F.when(F.col(name).isNotNull(), 1.0).otherwise(0.0) if name in df.columns else F.lit(0.0)

def build_header_features(header, lines):
    line_rollup = (lines.groupBy('InvoiceId')
        .agg(
            F.count('LineNumber').cast('double').alias('line_count'),
            F.sum(num(lines, 'ItemSubTotal')).cast('double').alias('sum_line_subtotal'),
            F.avg(num(lines, 'Quantity')).cast('double').alias('avg_quantity'),
            F.avg(num(lines, 'UnitPrice')).cast('double').alias('avg_unit_price'),
            F.max(num(lines, 'ItemSubTotal')).cast('double').alias('max_line_subtotal'),
            F.stddev(num(lines, 'ItemSubTotal')).cast('double').alias('stddev_line_subtotal')
        ))

    return (header.join(line_rollup, 'InvoiceId', 'left')
        .select(
            'InvoiceId',
            F.col('_source_file').cast('string').alias('SourceFile'),
            F.col('source_type').cast('string').alias('SourceType'),
            present(header, 'InvoiceId').alias('invoice_id_present'),
            present(header, 'OrderDate').alias('order_date_present'),
            present(header, 'CustomerName').alias('customer_name_present'),
            present(header, 'ShipMode').alias('ship_mode_present'),
            F.when(F.col('source_type') == 'csv', 1.0).when(F.col('source_type') == 'json', 2.0).otherwise(0.0).alias('source_type_code'),
            F.abs(F.xxhash64(F.coalesce(F.col('ShipMode'), F.lit('missing'))) % 1000).cast('double').alias('ship_mode_code'),
            num(header, 'BalanceDue').alias('BalanceDue'),
            num(header, 'SubTotal').alias('SubTotal'),
            num(header, 'DiscountPercent').alias('DiscountPercent'),
            num(header, 'DiscountAmount').alias('DiscountAmount'),
            num(header, 'ShippingAmount').alias('ShippingAmount'),
            num(header, 'InvoiceTotal').alias('InvoiceTotal'),
            F.coalesce(F.col('line_count'), F.lit(0.0)).alias('line_count'),
            F.coalesce(F.col('sum_line_subtotal'), F.lit(0.0)).alias('sum_line_subtotal'),
            F.coalesce(F.col('avg_quantity'), F.lit(0.0)).alias('avg_quantity'),
            F.coalesce(F.col('avg_unit_price'), F.lit(0.0)).alias('avg_unit_price'),
            F.coalesce(F.col('max_line_subtotal'), F.lit(0.0)).alias('max_line_subtotal'),
            F.coalesce(F.col('stddev_line_subtotal'), F.lit(0.0)).alias('stddev_line_subtotal'))
        .withColumn('calc_invoice_total', F.col('SubTotal') - F.coalesce(F.col('DiscountAmount'), F.lit(0.0)) + F.coalesce(F.col('ShippingAmount'), F.lit(0.0)))
        .withColumn('header_total_diff', F.col('InvoiceTotal') - F.col('calc_invoice_total'))
        .withColumn('header_subtotal_rollup_diff', F.col('SubTotal') - F.col('sum_line_subtotal'))
        .withColumn('discount_ratio', F.when(F.col('SubTotal') != 0, F.col('DiscountAmount') / F.col('SubTotal')).otherwise(0.0))
        .withColumn('missing_header_field_count', (1.0 - F.col('invoice_id_present')) + (1.0 - F.col('order_date_present')) + (1.0 - F.col('customer_name_present')) + (1.0 - F.col('ship_mode_present')))
        .withColumn('missing_financial_field_count',
            F.when(F.col('BalanceDue').isNull(), 1.0).otherwise(0.0)
            + F.when(F.col('SubTotal').isNull(), 1.0).otherwise(0.0)
            + F.when(F.col('DiscountPercent').isNull(), 1.0).otherwise(0.0)
            + F.when(F.col('DiscountAmount').isNull(), 1.0).otherwise(0.0)
            + F.when(F.col('ShippingAmount').isNull(), 1.0).otherwise(0.0)
            + F.when(F.col('InvoiceTotal').isNull(), 1.0).otherwise(0.0))
        .withColumn('abs_header_total_diff', F.abs(F.coalesce(F.col('header_total_diff'), F.lit(0.0))))
        .withColumn('abs_header_subtotal_rollup_diff', F.abs(F.coalesce(F.col('header_subtotal_rollup_diff'), F.lit(0.0))))
        .withColumn('shipping_ratio', F.when(F.col('SubTotal') != 0, F.col('ShippingAmount') / F.col('SubTotal')).otherwise(0.0))
        .withColumn('balance_due_ratio', F.when(F.col('InvoiceTotal') != 0, F.col('BalanceDue') / F.col('InvoiceTotal')).otherwise(0.0))
        .withColumn('line_count_log', F.log1p(F.abs(F.coalesce(F.col('line_count'), F.lit(0.0)))))
        .withColumn('avg_line_amount', F.when(F.col('line_count') != 0, F.col('sum_line_subtotal') / F.col('line_count')).otherwise(0.0)))

def build_line_features(header, lines):
    header_invoice_total = header.select(
        'InvoiceId',
        num(header, 'InvoiceTotal').alias('_invoice_total')
    )
    line_source = lines.join(header_invoice_total, 'InvoiceId', 'left')

    return (line_source.select(
            'InvoiceId',
            F.col('LineNumber').cast('string').alias('LineNumber'),
            F.col('_source_file').cast('string').alias('SourceFile'),
            F.col('source_type').cast('string').alias('SourceType'),
            present(line_source, 'InvoiceId').alias('invoice_id_present'),
            present(line_source, 'LineNumber').alias('line_number_present'),
            present(line_source, 'ProductName').alias('product_name_present'),
            present(line_source, 'ProductId').alias('product_id_present'),
            present(line_source, 'Quantity').alias('quantity_present'),
            present(line_source, 'UnitPrice').alias('unit_price_present'),
            present(line_source, 'ItemSubTotal').alias('item_subtotal_present'),
            F.when(F.col('source_type') == 'csv', 1.0).when(F.col('source_type') == 'json', 2.0).otherwise(0.0).alias('source_type_code'),
            F.abs(F.xxhash64(F.coalesce(F.col('ProductName'), F.lit('missing'))) % 1000).cast('double').alias('product_name_code'),
            num(line_source, 'Quantity').alias('Quantity'),
            num(line_source, 'UnitPrice').alias('UnitPrice'),
            num(line_source, 'ItemSubTotal').alias('ItemSubTotal'),
            F.coalesce(F.col('_invoice_total'), F.lit(0.0)).alias('_invoice_total'))
        .withColumn('calc_item_subtotal', F.col('Quantity') * F.col('UnitPrice'))
        .withColumn('line_subtotal_diff', F.col('ItemSubTotal') - F.col('calc_item_subtotal'))
        .withColumn('abs_line_subtotal_diff', F.abs(F.coalesce(F.col('line_subtotal_diff'), F.lit(0.0))))
        .withColumn('line_amount_ratio_to_invoice', F.when(F.col('_invoice_total') != 0, F.col('ItemSubTotal') / F.col('_invoice_total')).otherwise(0.0))
        .withColumn('quantity_log', F.log1p(F.abs(F.coalesce(F.col('Quantity'), F.lit(0.0)))))
        .withColumn('unit_price_log', F.log1p(F.abs(F.coalesce(F.col('UnitPrice'), F.lit(0.0)))))
        .withColumn('item_subtotal_log', F.log1p(F.abs(F.coalesce(F.col('ItemSubTotal'), F.lit(0.0)))))
        .drop('_invoice_total'))

def score(df, model_name, id_cols):
    model, meta = load_model(model_name)
    features = meta['feature_columns']
    missing_ids = [c for c in id_cols if c not in df.columns]
    missing_features = [c for c in features if c not in df.columns]
    if missing_ids or missing_features:
        raise RuntimeError(
            f'{model_name} scoring input schema mismatch. '
            f'Missing id columns={missing_ids}; missing feature columns={missing_features}'
        )

    pdf = df.select(*(id_cols + features)).fillna(0.0).toPandas()
    if len(pdf) == 0:
        raise RuntimeError(f'{model_name} scoring input has no rows')

    x = pdf[features].astype('float32').values
    x = ((x - np.array(meta['mean'], dtype='float32')) / np.array(meta['std'], dtype='float32')).astype('float32')
    with torch.no_grad():
        t = torch.tensor(x, dtype=torch.float32)
        scores = torch.mean((model(t) - t) ** 2, dim=1).numpy()
    pdf['anomaly_score'] = scores.astype('float64')
    pdf['threshold'] = float(meta['threshold'])
    pdf['is_anomaly'] = pdf['anomaly_score'] >= pdf['threshold']
    pdf['model_name'] = model_name
    pdf['model_version'] = MODEL_VERSION
    pdf['score_run_id'] = RUN_ID
    return spark.createDataFrame(pdf[id_cols + ['anomaly_score','threshold','is_anomaly','model_name','model_version','score_run_id']])


def explain_feature_contributions(df, model_name, id_cols, dataset, top_n=5):
    model, meta = load_model(model_name)
    features = meta['feature_columns']
    pdf = df.select(*(id_cols + features)).fillna(0.0).toPandas()
    if len(pdf) == 0:
        schema = 'layer string, dataset string, InvoiceId string, LineNumber string, SourceFile string, SourceType string, model_name string, model_version string, score_run_id string, anomaly_score double, threshold double, feature_name string, feature_value double, training_mean double, training_std double, standardized_value double, reconstruction_value double, feature_contribution double, contribution_share double, contribution_rank int'
        return spark.createDataFrame([], schema)

    raw = pdf[features].astype('float32').values
    mean = np.array(meta['mean'], dtype='float32')
    std = np.array(meta['std'], dtype='float32')
    x = ((raw - mean) / std).astype('float32')
    with torch.no_grad():
        t = torch.tensor(x, dtype=torch.float32)
        recon = model(t).numpy()
    contrib = (recon - x) ** 2
    scores = contrib.mean(axis=1)
    rows = []
    for i, row in pdf.iterrows():
        order = np.argsort(contrib[i])[::-1][:top_n]
        total = float(contrib[i].sum())
        for rank, j in enumerate(order, start=1):
            out = {c: row[c] for c in id_cols}
            out.update({
                'layer': 'silver',
                'dataset': dataset,
                'LineNumber': '' if dataset == 'header' else str(row.get('LineNumber', '')),
                'model_name': model_name,
                'model_version': MODEL_VERSION,
                'score_run_id': RUN_ID,
                'anomaly_score': float(scores[i]),
                'threshold': float(meta['threshold']),
                'feature_name': features[j],
                'feature_value': float(raw[i, j]),
                'training_mean': float(mean[j]),
                'training_std': float(std[j]),
                'standardized_value': float(x[i, j]),
                'reconstruction_value': float(recon[i, j]),
                'feature_contribution': float(contrib[i, j]),
                'contribution_share': float(contrib[i, j] / total) if total else None,
                'contribution_rank': int(rank),
            })
            rows.append(out)
    return spark.createDataFrame(rows)



In [0]:
header = read_delta(SILVER_HEADER_PATH, 'silver_header')
lines = read_delta(SILVER_LINES_PATH, 'silver_lines')

header_features = build_header_features(header, lines)
line_features = build_line_features(header, lines)
print('Built current silver scoring features from standardized header and line paths')

header_features.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(SCORING_HEADER_FEATURE_PATH)
line_features.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(SCORING_LINES_FEATURE_PATH)

header_scores = score(header_features, 'silver_header_autoencoder', ['InvoiceId','SourceFile','SourceType'])
line_scores = score(line_features, 'silver_lines_autoencoder', ['InvoiceId','LineNumber','SourceFile','SourceType'])

header_scores.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(HEADER_SCORE_PATH)
line_scores.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(LINES_SCORE_PATH)
display(header_scores.orderBy(F.desc('anomaly_score')).limit(20))
display(line_scores.orderBy(F.desc('anomaly_score')).limit(20))


Built current silver scoring features from standardized header and line paths


InvoiceId,SourceFile,SourceType,anomaly_score,threshold,is_anomaly,model_name,model_version,score_run_id
20126,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,2.054250274018684E30,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
37936,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Anthony%2520Jacobs_37936.pdf.ocr.json,json,1.6488729659540517E30,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
15953,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Zuschuss%2520Carroll_15953.pdf.ocr.json,json,1.4228783377397498E30,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
20958,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Brendan%2520Murry_20958.pdf.ocr.json,json,1.1131339377722336E30,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
18994,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Carlos%2520Soltero_18994.pdf.ocr.json,json,8.333461907097219E29,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
8367,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Michael%2520Stewart_8367.pdf.ocr.json,json,5.242886495024062E29,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
14984,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Denise%2520Leinenbach_14984.pdf.ocr.json,json,4.178919192546237E29,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
2642,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Larry%2520Hughes_2642.pdf.ocr.json,json,3.796382573338766E29,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
18672,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Zuschuss%2520Donatelli_18672.pdf.ocr.json,json,3.2060754293005015E29,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452
2371,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Juliana%2520Krohn_2371.pdf.ocr.json,json,2.890025952278649E29,0.15282315015792847,true,silver_header_autoencoder,v1,20260425113452


InvoiceId,LineNumber,SourceFile,SourceType,anomaly_score,threshold,is_anomaly,model_name,model_version,score_run_id
20025,1,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,3.4700615745536E13,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20022,1,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,2.6386416795648E13,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20178,4,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,2.0935633534976E13,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20182,4,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,1.554255970304E13,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20048,4,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,1.231491301376E12,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20208,1,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,4.78457069568E11,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20063,2,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,1.3704359936E11,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20026,2,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,3.4166659072E10,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20126,3,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,6.2326744E7,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452
20221,2,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,3.581348E7,8.352589793503284E-4,true,silver_lines_autoencoder,v1,20260425113452


In [0]:
header_anomaly_features = header_features.join(
    header_scores.filter('is_anomaly').select('InvoiceId', 'SourceFile', 'SourceType').dropDuplicates(),
    ['InvoiceId', 'SourceFile', 'SourceType'],
    'inner'
)
line_anomaly_features = line_features.join(
    line_scores.filter('is_anomaly').select('InvoiceId', 'LineNumber', 'SourceFile', 'SourceType').dropDuplicates(),
    ['InvoiceId', 'LineNumber', 'SourceFile', 'SourceType'],
    'inner'
)

header_feature_contributions = explain_feature_contributions(
    header_anomaly_features,
    'silver_header_autoencoder',
    ['InvoiceId', 'SourceFile', 'SourceType'],
    dataset='header',
    top_n=5
)
line_feature_contributions = explain_feature_contributions(
    line_anomaly_features,
    'silver_lines_autoencoder',
    ['InvoiceId', 'LineNumber', 'SourceFile', 'SourceType'],
    dataset='lines',
    top_n=5
)

feature_contributions = (header_feature_contributions
    .unionByName(line_feature_contributions, allowMissingColumns=True)
    .select(
        'layer', 'dataset', F.col('InvoiceId').cast('string').alias('InvoiceId'),
        F.col('LineNumber').cast('string').alias('LineNumber'),
        F.col('SourceFile').cast('string').alias('SourceFile'),
        F.col('SourceType').cast('string').alias('SourceType'),
        'model_name', 'model_version', 'score_run_id', 'anomaly_score', 'threshold',
        'feature_name', 'feature_value', 'training_mean', 'training_std',
        'standardized_value', 'reconstruction_value', 'feature_contribution',
        'contribution_share', 'contribution_rank'
    ))

feature_contributions.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(AIOPS_FEATURE_CONTRIBUTION_PATH)
dbutils.jobs.taskValues.set(key='aiops_feature_contribution_path', value=AIOPS_FEATURE_CONTRIBUTION_PATH)
dbutils.jobs.taskValues.set(key='aiops_scoring_header_feature_path', value=SCORING_HEADER_FEATURE_PATH)
dbutils.jobs.taskValues.set(key='aiops_scoring_lines_feature_path', value=SCORING_LINES_FEATURE_PATH)
display(feature_contributions.orderBy('dataset', 'InvoiceId', 'LineNumber', 'contribution_rank'))


layer,dataset,InvoiceId,LineNumber,SourceFile,SourceType,model_name,model_version,score_run_id,anomaly_score,threshold,feature_name,feature_value,training_mean,training_std,standardized_value,reconstruction_value,feature_contribution,contribution_share,contribution_rank
silver,header,10052,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Jill%2520Matthias_10052.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.6058819890022278,0.15282315015792847,stddev_line_subtotal,0.0,1140.37646484375,1156.0745849609375,-0.9864211678504944,1.812455415725708,7.833709716796875,0.4309810400009155,1
silver,header,10052,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Jill%2520Matthias_10052.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.6058819890022278,0.15282315015792847,avg_unit_price,849.7999877929688,245.14303588867188,174.40896606445312,3.4668915271759033,1.672248125076294,3.2207448482513428,0.17719317972660065,2
silver,header,10052,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Jill%2520Matthias_10052.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.6058819890022278,0.15282315015792847,avg_line_amount,4249.0,1109.964599609375,985.9457397460938,3.183781147003174,2.062854766845703,1.2564759254455566,0.06912654638290405,3
silver,header,10052,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Jill%2520Matthias_10052.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.6058819890022278,0.15282315015792847,source_type_code,2.0,1.0,1.0,1.0,0.00888168066740036,0.9823154807090759,0.054043278098106384,4
silver,header,10052,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Jill%2520Matthias_10052.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.6058819890022278,0.15282315015792847,avg_quantity,5.0,4.488586902618408,1.3574048280715942,0.37675797939300537,-0.4794566035270691,0.7331033945083618,0.04033257067203522,5
silver,header,10214,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Rick%2520Reed_10214.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.9363434910774231,0.15282315015792847,stddev_line_subtotal,0.0,1140.37646484375,1156.0745849609375,-0.9864211678504944,2.109379768371582,9.583983421325684,0.34118473529815674,1
silver,header,10214,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Rick%2520Reed_10214.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.9363434910774231,0.15282315015792847,ship_mode_code,75.0,329.72955322265625,173.6450958251953,-1.4669550657272339,0.5770542621612549,4.177973747253418,0.14873366057872772,2
silver,header,10214,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Rick%2520Reed_10214.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.9363434910774231,0.15282315015792847,avg_quantity,5.0,4.488586902618408,1.3574048280715942,0.37675797939300537,-1.2147307395935059,2.532836437225342,0.09016763418912888,3
silver,header,10214,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Rick%2520Reed_10214.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.9363434910774231,0.15282315015792847,abs_header_total_diff,4.547473508864641E-13,1.6167010221876388E-13,3.441328981761832E-13,0.8516397476196289,-0.6988343596458435,2.403970241546631,0.08558006584644318,4
silver,header,10214,,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Rick%2520Reed_10214.pdf.ocr.json,json,silver_header_autoencoder,v1,20260425113452,0.9363434910774231,0.15282315015792847,avg_unit_price,848.3200073242188,245.14303588867188,174.40896606445312,3.4584059715270996,2.15690279006958,1.6939104795455933,0.06030231714248657,5


In [0]:
def severity_expr():
    return F.when(F.col('anomaly_score') >= F.col('threshold') * 1.5, 'ERROR').otherwise('WARNING')

header_issues = (
    header_scores.filter('is_anomaly')
    .select(
        F.lit('silver').alias('layer'),
        F.lit('header').alias('dataset'),
        F.lit('aiops_silver_header_autoencoder').alias('detector_id'),
        'model_name',
        'model_version',
        severity_expr().alias('severity'),
        'SourceFile',
        'SourceType',
        F.col('InvoiceId').cast('string'),
        F.lit(None).cast('string').alias('LineNumber'),
        'anomaly_score',
        'threshold',
        F.lit('Silver header has abnormal learned feature pattern').alias('dq_reason'),
        F.current_timestamp().alias('issue_ts'),
        F.lit(RUN_ID).alias('run_id')
    )
)

line_issues = (
    line_scores.filter('is_anomaly')
    .select(
        F.lit('silver').alias('layer'),
        F.lit('lines').alias('dataset'),
        F.lit('aiops_silver_lines_autoencoder').alias('detector_id'),
        'model_name',
        'model_version',
        severity_expr().alias('severity'),
        'SourceFile',
        'SourceType',
        F.col('InvoiceId').cast('string'),
        F.col('LineNumber').cast('string'),
        'anomaly_score',
        'threshold',
        F.lit('Silver line has abnormal learned feature pattern').alias('dq_reason'),
        F.current_timestamp().alias('issue_ts'),
        F.lit(RUN_ID).alias('run_id')
    )
)

issues = header_issues.unionByName(line_issues).dropDuplicates()
issues.write.format('delta').mode('append').save(AIOPS_ISSUE_LOG_PATH)
display(issues.orderBy('severity', F.desc('anomaly_score')))

layer,dataset,detector_id,model_name,model_version,severity,SourceFile,SourceType,InvoiceId,LineNumber,anomaly_score,threshold,dq_reason,issue_ts,run_id
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://csv-invoices@hantstorageaccount.dfs.core.windows.net/invoices_1000_items.csv,csv,20126,null,2.054250274018684E30,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Anthony%2520Jacobs_37936.pdf.ocr.json,json,37936,null,1.6488729659540517E30,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Zuschuss%2520Carroll_15953.pdf.ocr.json,json,15953,null,1.4228783377397498E30,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Brendan%2520Murry_20958.pdf.ocr.json,json,20958,null,1.1131339377722336E30,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Carlos%2520Soltero_18994.pdf.ocr.json,json,18994,null,8.333461907097219E29,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Michael%2520Stewart_8367.pdf.ocr.json,json,8367,null,5.242886495024062E29,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Denise%2520Leinenbach_14984.pdf.ocr.json,json,14984,null,4.178919192546237E29,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Larry%2520Hughes_2642.pdf.ocr.json,json,2642,null,3.796382573338766E29,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Zuschuss%2520Donatelli_18672.pdf.ocr.json,json,18672,null,3.2060754293005015E29,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452
silver,header,aiops_silver_header_autoencoder,silver_header_autoencoder,v1,ERROR,abfss://di-json-invoices@hantstorageaccount.dfs.core.windows.net/invoice_Juliana%2520Krohn_2371.pdf.ocr.json,json,2371,null,2.890025952278649E29,0.15282315015792847,Silver header has abnormal learned feature pattern,2026-04-25T11:35:21.988915Z,20260425113452


In [0]:
header = spark.read.format('delta').load(SILVER_HEADER_PATH)
lines = spark.read.format('delta').load(SILVER_LINES_PATH)

error_ids = issues.filter((F.col('severity') == 'ERROR') & F.col('InvoiceId').isNotNull()).select('InvoiceId').distinct()
warning_ids = issues.filter((F.col('severity') == 'WARNING') & F.col('InvoiceId').isNotNull()).select('InvoiceId').distinct().join(error_ids, 'InvoiceId', 'left_anti')

header.join(error_ids, 'InvoiceId', 'inner').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(AIOPS_QUARANTINE_HEADER_PATH)
lines.join(error_ids, 'InvoiceId', 'inner').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(AIOPS_QUARANTINE_LINES_PATH)
header.join(warning_ids, 'InvoiceId', 'inner').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(AIOPS_WARNING_HEADER_PATH)
lines.join(warning_ids, 'InvoiceId', 'inner').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(AIOPS_WARNING_LINES_PATH)

dbutils.jobs.taskValues.set(key='aiops_silver_issue_log_path', value=AIOPS_ISSUE_LOG_PATH)
dbutils.jobs.taskValues.set(key='aiops_silver_header_score_path', value=HEADER_SCORE_PATH)
dbutils.jobs.taskValues.set(key='aiops_silver_lines_score_path', value=LINES_SCORE_PATH)

In [0]:
ended_at = datetime.now(timezone.utc)
rows_evaluated = int(header_scores.count() + line_scores.count())
anomalies_detected = int(issues.count())
runtime_seconds = (ended_at - NOTEBOOK_STARTED_AT).total_seconds()
records_per_second = float(rows_evaluated / runtime_seconds) if runtime_seconds > 0 else None

runtime_df = spark.createDataFrame([(
    'silver',
    'aiops_score_silver_end_to_end',
    RUN_ID,
    MODEL_VERSION,
    NOTEBOOK_STARTED_AT,
    ended_at,
    float(runtime_seconds),
    rows_evaluated,
    anomalies_detected,
    records_per_second,
    datetime.now(timezone.utc),
)], AIOPS_RUNTIME_SCHEMA)

runtime_df.write.format('delta').mode('append').save(AIOPS_RUNTIME_METRICS_PATH)
spark.sql('CREATE SCHEMA IF NOT EXISTS `hant-catalog`.invoice')
spark.sql(f'''CREATE TABLE IF NOT EXISTS {AIOPS_RUNTIME_METRICS_TABLE}
USING DELTA
LOCATION "{AIOPS_RUNTIME_METRICS_PATH}"''')

dbutils.jobs.taskValues.set(key='aiops_runtime_metrics_path', value=AIOPS_RUNTIME_METRICS_PATH)
display(runtime_df)


layer,operation_name,run_id,model_version,started_at,ended_at,runtime_seconds,rows_evaluated,anomalies_detected,records_per_second,recorded_at
silver,aiops_score_silver_end_to_end,20260425113452,v1,2026-04-25T11:34:53.115376Z,2026-04-25T11:35:32.518996Z,39.40362,3252,2169,82.53048831554057,2026-04-25T11:35:33.739547Z
